In [1]:
import sys
!{sys.executable} -m pip install matplotlib scikit-learn pandas numpy joblib catboost

  Using cached matplotlib-3.11.1-cp314-cp314-win_amd64.whl.metadata (80 kB)
  Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl.metadata (1.5 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp314-cp314-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.9.0-py3-none-any.whl.metadata (9.0 kB)
Using cached matplotlib-3.11.1-cp314-cp314-win_amd64.whl (9.5 MB)
Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl (101.7 MB)
Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl (232 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp314-cp314-win_amd64.whl (2.3 MB)
Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl (75 kB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
Us

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install -q xgboost lightgbm catboost


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import joblib
import os

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("Libraries imported successfully")

Libraries imported successfully


In [5]:
df = pd.read_csv("D:\hospital_dashboard\dataset\diabetic_data.csv")

print("Dataset shape:", df.shape)

df.head()

<>:1: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
<>:1: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
C:\Users\vishnuvardhan\AppData\Local\Temp\ipykernel_92044\780056841.py:1: SyntaxWarning: "\h" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\h"? A raw string is also an option.
  df = pd.read_csv("D:\hospital_dashboard\dataset\diabetic_data.csv")


Dataset shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [6]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (101766, 50)

Columns:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

Missing values:
max_glu_serum               96420
A1Cresult                   84748
race                            0
gender                          0

In [7]:
drop_columns = [
    "weight",
    "payer_code",
    "medical_specialty",
    "encounter_id"
]

df = df.drop(
    columns=[col for col in drop_columns if col in df.columns],
    errors="ignore"
)

print("Remaining shape:", df.shape)

Remaining shape: (101766, 46)


In [8]:
if "patient_nbr" in df.columns:

    df = df.drop_duplicates(
        subset=["patient_nbr"]
    )

    df = df.drop(
        columns=["patient_nbr"]
    )

print("Shape after removing duplicate patients:", df.shape)

Shape after removing duplicate patients: (71518, 45)


In [9]:
# Create binary target
# <30 = Readmitted
# NO and >30 = Not Readmitted

df["target"] = (
    df["readmitted"] == "<30"
).astype(int)

print(df["target"].value_counts())

print(
    df["target"].value_counts(
        normalize=True
    ) * 100
)

target
0    65225
1     6293
Name: count, dtype: int64
target
0    91.200817
1     8.799183
Name: proportion, dtype: float64


In [10]:
id_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in id_columns:

    if col in df.columns:

        df[col] = df[col].astype(str)

print("ID columns converted to categorical strings")

ID columns converted to categorical strings


In [11]:
def convert_age(age):

    if pd.isna(age):
        return np.nan

    age = str(age)

    if "[" in age and "-" in age:

        values = age.strip("[]()").split("-")

        try:
            lower = int(values[0])
            upper = int(values[1])

            return (lower + upper) / 2

        except:
            return np.nan

    return np.nan


if "age" in df.columns:

    df["age"] = df["age"].apply(
        convert_age
    )

print(df["age"].head())

0     5.0
1    15.0
2    25.0
3    35.0
4    45.0
Name: age, dtype: float64


In [12]:
X = df.drop(
    columns=["readmitted", "target"],
    errors="ignore"
)

y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (71518, 44)
y shape: (71518,)

Target distribution:
target
0    65225
1     6293
Name: count, dtype: int64


In [13]:
feature_names = X.columns.tolist()

print("Number of features:", len(feature_names))

print(feature_names)

Number of features: 44
['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [14]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print("Training samples:", len(X_train))

print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

Training samples: 57214
Testing samples: 14304

Training distribution:
target
0    52180
1     5034
Name: count, dtype: int64

Testing distribution:
target
0    13045
1     1259
Name: count, dtype: int64


In [15]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical features:
['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [16]:
numeric_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )

    ]
)


categorical_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )

    ]
)


preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            numeric_pipeline,
            numeric_features
        ),

        (
            "cat",
            categorical_pipeline,
            categorical_features
        )

    ]
)

print("Preprocessor created successfully")

Preprocessor created successfully


In [17]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print(
    "Processed training shape:",
    X_train_processed.shape
)

print(
    "Processed testing shape:",
    X_test_processed.shape
)

Processed training shape: (57214, 2228)
Processed testing shape: (14304, 2228)


In [18]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_processed
)

X_test_scaled = scaler.transform(
    X_test_processed
)

print("Scaling completed")

Scaling completed


In [19]:
# ============================================================
# FINAL MODEL TRAINING + ARTIFACT EXPORT FOR HEALTHFORECAST AI
# ============================================================

import os
import pickle
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# ============================================================
# 1. CREATE ML MODELS DIRECTORY
# ============================================================

MODEL_DIR = r"D:\hospital\backend\ml_models"

os.makedirs(MODEL_DIR, exist_ok=True)

print("ML model directory:", MODEL_DIR)


# ============================================================
# 2. TRAIN FINAL CATBOOST MODEL
# ============================================================

print("\nTraining CatBoost model...")

catboost_model = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=0
)

catboost_model.fit(
    X_train_scaled,
    y_train
)

print("CatBoost training completed!")


# ============================================================
# 3. EVALUATE MODEL
# ============================================================

y_pred = catboost_model.predict(
    X_test_scaled
)

y_prob = catboost_model.predict_proba(
    X_test_scaled
)[:, 1]


accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)


print("\n============================================================")
print("FINAL CATBOOST MODEL RESULTS")
print("============================================================")

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ============================================================
# 4. GET FINAL FEATURE NAMES
# ============================================================

try:

    transformed_feature_names = (
        preprocessor.get_feature_names_out()
    )

    print(
        "\nTransformed feature count:",
        len(transformed_feature_names)
    )

except Exception:

    transformed_feature_names = X.columns.tolist()

    print(
        "\nCould not get transformed feature names."
    )


# ============================================================
# 5. SAVE ALL ML ARTIFACTS USING PICKLE
# ============================================================

model_path = os.path.join(
    MODEL_DIR,
    "model.pkl"
)

preprocessor_path = os.path.join(
    MODEL_DIR,
    "preprocessor.pkl"
)

scaler_path = os.path.join(
    MODEL_DIR,
    "scaler.pkl"
)

feature_names_path = os.path.join(
    MODEL_DIR,
    "feature_names.pkl"
)


# Save model
with open(
    model_path,
    "wb"
) as f:

    pickle.dump(
        catboost_model,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# Save preprocessor
with open(
    preprocessor_path,
    "wb"
) as f:

    pickle.dump(
        preprocessor,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# Save scaler
with open(
    scaler_path,
    "wb"
) as f:

    pickle.dump(
        scaler,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# Save feature names
with open(
    feature_names_path,
    "wb"
) as f:

    pickle.dump(
        list(transformed_feature_names),
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# ============================================================
# 6. VERIFY SAVED FILES
# ============================================================

print("\n============================================================")
print("ML ARTIFACTS SAVED SUCCESSFULLY")
print("============================================================")

print("Model:")
print(model_path)

print("\nPreprocessor:")
print(preprocessor_path)

print("\nScaler:")
print(scaler_path)

print("\nFeature Names:")
print(feature_names_path)


# ============================================================
# 7. VERIFY FILE SIZES
# ============================================================

print("\n============================================================")
print("FILE VERIFICATION")
print("============================================================")

for path in [
    model_path,
    preprocessor_path,
    scaler_path,
    feature_names_path
]:

    if os.path.exists(path):

        print(
            "OK:",
            os.path.basename(path),
            "->",
            os.path.getsize(path),
            "bytes"
        )

    else:

        print(
            "ERROR: File not found:",
            path
        )


# ============================================================
# 8. TEST LOADING ALL ARTIFACTS
# ============================================================

print("\n============================================================")
print("TESTING ARTIFACT LOADING")
print("============================================================")

with open(model_path, "rb") as f:

    test_model = pickle.load(f)


with open(preprocessor_path, "rb") as f:

    test_preprocessor = pickle.load(f)


with open(scaler_path, "rb") as f:

    test_scaler = pickle.load(f)


with open(feature_names_path, "rb") as f:

    test_feature_names = pickle.load(f)


print("Model loaded successfully:")
print(type(test_model))

print("\nPreprocessor loaded successfully:")
print(type(test_preprocessor))

print("\nScaler loaded successfully:")
print(type(test_scaler))

print("\nFeature names loaded successfully:")
print(type(test_feature_names))

print(
    "\nNumber of saved feature names:",
    len(test_feature_names)
)

print("\n============================================================")
print("ALL 4 ML ARTIFACTS ARE VALID!")
print("============================================================")

ML model directory: D:\hospital\backend\ml_models

Training CatBoost model...
CatBoost training completed!

FINAL CATBOOST MODEL RESULTS
Accuracy : 0.6736
Precision: 0.1454
Recall   : 0.5552
F1 Score : 0.2304
ROC-AUC  : 0.6652

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.69      0.79     13045
           1       0.15      0.56      0.23      1259

    accuracy                           0.67     14304
   macro avg       0.54      0.62      0.51     14304
weighted avg       0.87      0.67      0.74     14304


Transformed feature count: 2228

ML ARTIFACTS SAVED SUCCESSFULLY
Model:
D:\hospital\backend\ml_models\model.pkl

Preprocessor:
D:\hospital\backend\ml_models\preprocessor.pkl

Scaler:
D:\hospital\backend\ml_models\scaler.pkl

Feature Names:
D:\hospital\backend\ml_models\feature_names.pkl

FILE VERIFICATION
OK: model.pkl -> 325718 bytes
OK: preprocessor.pkl -> 17710 bytes
OK: scaler.pkl -> 53884 bytes
OK: feature_names.